# Stage 00a.1 — Dataset Audit

**What this notebook does:**  
Audits the `raw/` folder against the PatSeer Excel export and produces:

| Output | Description |
|--------|-------------|
| `1639_DS_audit.xlsx` | Full per-patent status: matched, missing, extra, name mismatches |
| `patseer_missing_prompt.txt` | Ready-to-paste `PNC:(...)` queries for PatSeer re-search |

**Statuses assigned to each Excel record:**

| Status | Meaning |
|--------|---------|
| `OK` | Folder found, name matches after normalization |
| `MISSING from folder` | No folder found — needs to be downloaded |
| `DUPLICATE folders` | Two or more folders normalize to the same patent ID |

**Name normalization rules** (applied to both Excel IDs and folder names):  
- Strip trailing `/`, `PAFP` suffix, kind codes (`A1`, `B2`, …)  
- For US patents with a 4-digit year: `US2022267016A1` → `US2022_267016`  
- `record_XXXX` identifiers are kept as-is  

**Where it fits in the pipeline:**
```
00a.1  ← YOU ARE HERE  (audit what is downloaded vs what is in Excel)
 ↓
00a    (download missing patents from PatSeer using generated PNC query)
 ↓
00b    (positional matching → _F / _Fu labels)
```

---

| Cell | What it does |
|------|--------------|
| 1 | Load config and paths |
| 2 | Run audit — compare Excel vs `raw/` folders |
| 3 | Write `1639_DS_audit.xlsx` |
| 4 | Print summary + generate PatSeer PNC prompt |

In [1]:
import sys, importlib.util
from pathlib import Path

_cwd = Path().resolve()
repo_root = _cwd if (_cwd / "src").exists() else _cwd.parent
src_dir   = repo_root / "src"

_cl_path = src_dir / "config_loader.py"
_cl_spec = importlib.util.spec_from_file_location("config_loader", _cl_path)
_cl_mod  = importlib.util.module_from_spec(_cl_spec)
sys.modules["config_loader"] = _cl_mod
_cl_spec.loader.exec_module(_cl_mod)
load_config = _cl_mod.load_config

cfg = load_config()

RAW_DIR       = Path(cfg["paths"]["raw_images"])
EXCEL_PATH    = Path(cfg["paths"]["patseer_excel"])
OUTPUT_DIR    = repo_root / "notebooks" / "outputs"
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_EXCEL  = OUTPUT_DIR / "1639_DS_audit.xlsx"
OUTPUT_PROMPT = OUTPUT_DIR / "patseer_missing_prompt.txt"

print(f"Raw folder : {RAW_DIR}")
print(f"Excel      : {EXCEL_PATH}")
print(f"Outputs    : {OUTPUT_DIR}")

Raw folder : /mnt/storage_11tb/Drive_files_to_syncronize/3 - Images DataSets & Labelling Outputs/1639_DS/raw
Excel      : /mnt/storage_11tb/Drive_files_to_syncronize/2 - Patente & Validation/3 -Raw_Patent_Exports_PatSeer_&Gold_Standard/1639__dataset_08_06_26.xlsx
Outputs    : /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools/notebooks/outputs


In [2]:
import re
import pandas as pd
from collections import Counter


def normalize_id(raw: str) -> str:
    """
    Normalize a patent/record ID for cross-matching.
    Same rules applied to both Excel Record Numbers and folder names.
    """
    pid = str(raw).strip().rstrip("/")
    pid = re.sub(r"PAFP$", "", pid)

    if re.match(r"^record_\d+$", pid):
        return pid

    # US + 4-digit year pattern
    us_match = re.match(r"^(US)(\d{4})0*(\d+)[A-Za-z0-9]*$", pid)
    if us_match:
        country, year, core = us_match.groups()
        return f"{country}{year}_{core}"

    # Strip trailing kind codes (A1, B2, A, B, ...)
    pid = re.sub(r"[A-Z][0-9]$", "", pid)
    pid = re.sub(r"[A-Z]$", "", pid)
    return pid


# ── Load Excel ──────────────────────────────────────────────────────────────
df_excel = pd.read_excel(EXCEL_PATH)
id_col = df_excel.columns[0]  # "Record Number"

excel_records = []
for row_idx, val in enumerate(df_excel[id_col]):
    if pd.isna(val):
        continue
    rec = str(val).strip()
    excel_records.append({"excel_record": rec, "excel_row": row_idx + 2, "norm": normalize_id(rec)})

# ── Load raw folders ────────────────────────────────────────────────────────
raw_folders = []
for entry in sorted(RAW_DIR.iterdir()):
    if entry.is_dir():
        name = entry.name
        imgs = list(entry.glob("*.png")) + list(entry.glob("*.jpg")) + list(entry.glob("*.jpeg"))
        raw_folders.append({
            "folder_name": name,
            "norm": normalize_id(name),
            "image_count": len(imgs),
            "has_images": len(imgs) > 0,
        })

print(f"Excel records : {len(excel_records)}")
print(f"Raw folders   : {len(raw_folders)}")
print(f"Folders with 0 images : {sum(1 for f in raw_folders if not f['has_images'])}")

/home/vasco/anaconda3/envs/doclayout_yolo/lib/python3.10/site-packages/openpyxl/styles/stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")


Excel records : 1639
Raw folders   : 1156
Folders with 0 images : 0


In [3]:
# ── Build lookup tables ──────────────────────────────────────────────────────
excel_norm_map = {}
for r in excel_records:
    excel_norm_map.setdefault(r["norm"], []).append(r)

raw_norm_map = {}
for f in raw_folders:
    raw_norm_map.setdefault(f["norm"], []).append(f)

excel_norms = set(excel_norm_map.keys())

# ── Classify each Excel record ───────────────────────────────────────────────
rows_audit = []
missing_from_folder = []

for r in excel_records:
    norm = r["norm"]
    matched = raw_norm_map.get(norm, [])

    if len(matched) == 0:
        status = "MISSING from folder"
        missing_from_folder.append(r["excel_record"])
        folder_names = ""
        image_counts = ""
    elif len(matched) == 1:
        status = "OK"
        folder_names = matched[0]["folder_name"]
        image_counts = str(matched[0]["image_count"])
    else:
        status = "DUPLICATE folders"
        folder_names = " | ".join(f["folder_name"] for f in matched)
        image_counts = " | ".join(str(f["image_count"]) for f in matched)

    name_mismatch = ""
    for mf in matched:
        if mf["folder_name"] != r["excel_record"]:
            name_mismatch = f"Excel='{r['excel_record']}' vs Folder='{mf['folder_name']}'"

    rows_audit.append({
        "Excel Row":            r["excel_row"],
        "Excel Record Number":  r["excel_record"],
        "Normalized Key":       norm,
        "Status":               status,
        "Matched Folder(s)":    folder_names,
        "Image Count":          image_counts,
        "Name Mismatch":        name_mismatch,
    })

# ── Extra folders (in raw/ but not in Excel) ─────────────────────────────────
extra_folders = [
    {"Folder Name": f["folder_name"], "Image Count": f["image_count"]}
    for f in raw_folders if f["norm"] not in excel_norms
]

# ── Duplicate norm keys in raw folders ───────────────────────────────────────
raw_norm_counts = Counter(f["norm"] for f in raw_folders)
raw_dupes = [
    {"Folder Name": f["folder_name"], "Image Count": f["image_count"], "Normalized Key": f["norm"]}
    for f in raw_folders if raw_norm_counts[f["norm"]] > 1
]

print("Classification done.")
status_counts = Counter(r["Status"] for r in rows_audit)
for s, c in sorted(status_counts.items()):
    print(f"  {s}: {c}")
print(f"  Extra folders (not in Excel): {len(extra_folders)}")
print(f"  Duplicate norm keys in raw  : {len(raw_dupes)}")

Classification done.
  MISSING from folder: 586
  OK: 1053
  Extra folders (not in Excel): 103
  Duplicate norm keys in raw  : 0


In [4]:
df_audit  = pd.DataFrame(rows_audit)
df_extra  = pd.DataFrame(extra_folders)
df_rdupes = pd.DataFrame(raw_dupes)

summary_rows = [
    ("Total Excel records",                   len(excel_records)),
    ("Total raw folders",                      len(raw_folders)),
    ("Folders with 0 images",                  sum(1 for f in raw_folders if not f["has_images"])),
    ("OK (matched)",                           status_counts.get("OK", 0)),
    ("MISSING from folder",                    status_counts.get("MISSING from folder", 0)),
    ("DUPLICATE folders (same norm key)",      status_counts.get("DUPLICATE folders", 0)),
    ("Extra folders (not in Excel)",           len(extra_folders)),
    ("Duplicate norm keys in raw folders",     len(raw_dupes)),
]
df_summary = pd.DataFrame(summary_rows, columns=["Metric", "Count"])

with pd.ExcelWriter(OUTPUT_EXCEL, engine="openpyxl") as writer:
    df_summary.to_excel(writer, sheet_name="Summary",       index=False)
    df_audit.to_excel(  writer, sheet_name="Full Audit",    index=False)
    df_extra.to_excel(  writer, sheet_name="Extra Folders", index=False)
    if not df_rdupes.empty:
        df_rdupes.to_excel(writer, sheet_name="Raw Folder Duplicates", index=False)

print(f"Audit Excel written to: {OUTPUT_EXCEL}")

Audit Excel written to: /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools/notebooks/outputs/1639_DS_audit.xlsx


In [5]:
# ── Print summary ────────────────────────────────────────────────────────────
print("══════════════════════════════════════")
print("Dataset Audit Summary")
print("══════════════════════════════════════")
for metric, count in summary_rows:
    print(f"  {metric:<42}: {count}")
print("══════════════════════════════════════")

# ── Generate PatSeer PNC prompt ──────────────────────────────────────────────
if missing_from_folder:
    pnc_terms = []
    for rec in missing_from_folder:
        clean = re.sub(r"PAFP$", "", rec.strip())
        clean = re.sub(r"[A-Z][0-9]$", "", clean)
        pnc_terms.append(f'"{clean}"')

    CHUNK_SIZE = 200
    chunks = [pnc_terms[i:i+CHUNK_SIZE] for i in range(0, len(pnc_terms), CHUNK_SIZE)]

    lines = [
        f"# PatSeer PNC Query — {len(missing_from_folder)} missing patents",
        f"# Split into {len(chunks)} chunk(s) of up to {CHUNK_SIZE} terms\n",
    ]
    for i, chunk in enumerate(chunks, 1):
        query = "PNC:(" + " OR ".join(chunk) + ")"
        lines.append(f"# --- Chunk {i} ({len(chunk)} terms) ---")
        lines.append(query + "\n")

    OUTPUT_PROMPT.write_text("\n".join(lines))

    print(f"\nPatSeer prompt → {OUTPUT_PROMPT}")
    print(f"  {len(missing_from_folder)} missing patents split into {len(chunks)} query chunk(s)")
    print("\nFirst chunk preview:")
    print("PNC:(" + " OR ".join(pnc_terms[:5]) + " OR ... )")
else:
    print("\nNo missing patents — no PatSeer prompt generated.")

══════════════════════════════════════
Dataset Audit Summary
══════════════════════════════════════
  Total Excel records                       : 1639
  Total raw folders                         : 1156
  Folders with 0 images                     : 0
  OK (matched)                              : 1053
  MISSING from folder                       : 586
  DUPLICATE folders (same norm key)         : 0
  Extra folders (not in Excel)              : 103
  Duplicate norm keys in raw folders        : 0
══════════════════════════════════════

PatSeer prompt → /home/vasco/Vasco Workspace/Tese_Vasco_Lnx/Patent-Labelling-Tools/notebooks/outputs/patseer_missing_prompt.txt
  586 missing patents split into 3 query chunk(s)

First chunk preview:
PNC:("CA3096221" OR "CA2967402" OR "EP3119674" OR "USD902828S" OR "CN112478154A" OR ... )
